 **Node.js** with **PM2** , we can tweak the CI/CD setup so that:

- It pulls the latest code from the `staging` branch.
- Installs dependencies (`npm install`).
- Restarts the app using `pm2`.

---

## 🔐 Step-by-Step: CI/CD for Node.js with PM2 on EC2 (GitHub → EC2)

---

### ✅ 1. On Your EC2 Instance

#### ✅ A. Make sure Node.js, PM2, and Git are installed:
```bash
sudo apt update
sudo apt install git -y
curl -fsSL https://deb.nodesource.com/setup_18.x | sudo -E bash -
sudo apt install -y nodejs
sudo npm install -g pm2
```

#### ✅ B. Clone your private repo if not already:
```bash
cd /home/ubuntu/
git clone git@github.com:your-username/your-repo.git
cd your-repo
git checkout staging
npm install
pm2 start ecosystem.config.js  # or pm2 start app.js
```

> Make sure your app is configured in `pm2` with an ecosystem file or simple start command.

---

### ✅ 2. Generate SSH Keys (for GitHub Actions)

On your **local machine or dev environment**:
```bash
ssh-keygen -t rsa -b 4096 -C "github-cicd" -f github_ec2
```

You’ll get:
- `github_ec2` (private key) → for GitHub secret
- `github_ec2.pub` (public key) → goes to EC2

#### ✅ Copy public key to EC2:
```bash
cat github_ec2.pub | ssh ubuntu@your-ec2-ip 'cat >> ~/.ssh/authorized_keys'
```

---

### ✅ 3. Add GitHub Secrets

In your repo → **Settings → Secrets → Actions**:

| Secret Name       | Value                         |
|-------------------|-------------------------------|
| `EC2_SSH_KEY`     | Paste the content of `github_ec2` |
| `EC2_HOST`        | Your EC2 public IP            |
| `EC2_USER`        | `ubuntu`                      |
| `EC2_PROJECT_PATH`| `/home/ubuntu/your-repo`      |

---

### ✅ 4. GitHub Actions Workflow

Create `.github/workflows/deploy.yml` in your repo:
```yaml
name: Deploy to EC2 (Node + PM2)

on:
  push:
    branches:
      - staging

jobs:
  deploy:
    runs-on: ubuntu-latest

    steps:
    - name: Checkout code
      uses: actions/checkout@v3

    - name: Setup SSH
      uses: webfactory/ssh-agent@v0.8.0
      with:
        ssh-private-key: ${{ secrets.EC2_SSH_KEY }}

    - name: Deploy to EC2 and Restart App
      run: |
        ssh -o StrictHostKeyChecking=no ${{ secrets.EC2_USER }}@${{ secrets.EC2_HOST }} << 'EOF'
          cd ${{ secrets.EC2_PROJECT_PATH }}
          git fetch origin
          git reset --hard origin/staging
          npm install
          pm2 restart all
        EOF
```

---

### 🧠 Notes

- `git reset --hard origin/staging` ensures a clean pull.
- `pm2 restart all` restarts all apps (you can specify `pm2 restart your-app-name` if needed).
- Ensure your app is already configured in PM2 (`pm2 start app.js --name myapp`).
- To save pm2 config after reboot:
```bash
pm2 startup
pm2 save
```

---


In [ ]:
#

Great question!

---

## 📦 What is `ecosystem.config.js` in PM2?

The `ecosystem.config.js` file is a **configuration file** for PM2 that defines how to run your Node.js application.

It's **not mandatory**, but it's **highly recommended** because:

### ✅ Benefits:
- Keeps your **start/stop/restart** config in one place.
- Supports **multiple environments** (dev/staging/prod).
- Lets you configure:
  - App name
  - Script to run
  - Environment variables
  - Logs
  - Instances (for clustering)
  - Auto-restart options

---

## 📁 Example `ecosystem.config.js` for your case (Node + PM2 + staging):

```js
module.exports = {
  apps: [
    {
      name: "myapp-staging", // name shown in pm2 list
      script: "app.js", // or index.js or your main file
      instances: 1,
      autorestart: true,
      watch: false,
      max_memory_restart: "500M",
      env: {
        NODE_ENV: "development"
      },
      env_staging: {
        NODE_ENV: "staging"
      },
      env_production: {
        NODE_ENV: "production"
      }
    }
  ]
}
```

---

### 🔧 Usage with PM2

Start the app:
```bash
pm2 start ecosystem.config.js --env staging
```

Restart app:
```bash
pm2 restart ecosystem.config.js --env staging
```

Save the process list:
```bash
pm2 save
```

---

### 🧠 Optional Tip:
You can run different apps in same config file by adding more entries to the `apps` array.

---
